<a href="https://colab.research.google.com/github/TanmayB22122006/QuickCart-Stockout-Risk/blob/main/QuickCart_Stockout_Risk_Tanmay_Bokade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: QuickCart Stockout Risk
**Name:** Tanmay Purushottam Bokade  
**Roll Number:** TANMAY8604123  
**Batch:** July 2026  
**Date:** 20 September 2026

### Step 1: Data Loading & Sanity Checks
**Logic:**
- Loading all 5 datasets using `pd.read_csv()`.
- **Fixing the 'N/A' trap:** The suppliers table has missing values written as text `'N/A'`. I added `na_values=['N/A']` so pandas correctly reads them as `NaN` (nulls) instead of strings.
- **Row counts:** Printing the length of all tables to ensure data loaded properly.
- **Target check:** Checking the percentage of our target classes (Safe, At-Risk, Imminent) using `value_counts()`.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Loading Datasets ---
dim_stores = pd.read_csv('dim_stores.csv')
dim_skus = pd.read_csv('dim_skus.csv')
# Handling the literal 'N/A' trap explicitly
dim_suppliers = pd.read_csv('dim_suppliers.csv', na_values=['N/A'], keep_default_na=True)
dim_events = pd.read_csv('dim_events.csv')
fact_inventory = pd.read_csv('fact_inventory_daily.csv')

# --- Logic: Dynamic Sanity Checks (Row Counts) ---
print("--- Table Row Counts ---")
tables = {
    "dim_stores": dim_stores,
    "dim_skus": dim_skus,
    "dim_suppliers": dim_suppliers,
    "dim_events": dim_events,
    "fact_inventory_daily": fact_inventory
}

for name, df in tables.items():
    print(f"{name}: {len(df)} rows")

# --- Logic: Dynamic Target Class Distribution Check ---
print("\n--- Target Class Distribution (%) ---")
target_dist = (fact_inventory['stockout_risk'].value_counts(normalize=True) * 100).round(2)
print(target_dist)

--- Table Row Counts ---
dim_stores: 12 rows
dim_skus: 60 rows
dim_suppliers: 15 rows
dim_events: 30 rows
fact_inventory_daily: 21600 rows

--- Target Class Distribution (%) ---
stockout_risk
Safe        65.42
At-Risk     24.01
Imminent    10.57
Name: proportion, dtype: float64


### Step 2: Data Cleaning & Merging
**Logic:**
- **Clean Casing:** The `city_display` column has mixed cases (e.g., upper/lower case issues). I am using `.str.title()` to fix it so grouping works properly later.
- **Merge Tables:** Joining all 4 dimension tables (stores, skus, suppliers, events) with the main `fact_inventory` table using a left join.
- **Date Format:** Converting date columns to datetime format before merging the events table.

In [7]:
dim_stores['city_display'] = dim_stores['city_display'].str.title()

# Join all tables logically
df = fact_inventory.merge(dim_stores, on='store_id', how='left')
df = df.merge(dim_skus, on=['sku_id', 'supplier_id'], how='left')
df = df.merge(dim_suppliers, on='supplier_id', how='left')

df['date'] = pd.to_datetime(df['date'])
dim_events['date'] = pd.to_datetime(dim_events['date'])
df = df.merge(dim_events, on='date', how='left')

print("Shape after merge:", df.shape)

Shape after merge: (21600, 39)


### Step 3: Feature Engineering
**Logic:**
- **Imputation:** Filling missing supplier reliability scores with the median of their respective categories.
- **Derived Features:** Creating `reord_gap` and `cov_ratio` to give the model direct signals about stock urgency.
- **Temporal Features:** Extracting `day` from the date for time-series learning.

In [8]:
# Impute missing reliability with category median
df['rel_score'] = df.groupby('category')['reliability_score'].transform(lambda x: x.fillna(x.median()))
df.drop(columns=['reliability_score'], inplace=True)

df['reord_gap'] = df['reorder_point'] - df['closing_stock']
df['cov_ratio'] = df['days_of_cover'] / df['lead_time_days_expected']
df['day'] = df['date'].dt.day

print("Missing rel_score:", df['rel_score'].isnull().sum())
print("Current Shape:", df.shape)

Missing rel_score: 0
Current Shape: (21600, 42)


### Step 4: Encoding & Time-Based Split
**Logic:**
- **Target Mapping:** Assigning numeric values to `stockout_risk` (Safe=0, At-Risk=1, Imminent=2).
- **Dropping IDs:** Removing identifiers and highly missing columns (like `lead_time_days_actual`) that don't add predictive value.
- **Data Splitting:** To avoid data leakage, we split chronologically. Days <= 23 are used for training, and Days >= 24 for testing.

In [9]:
# Map target variable
target_map = {'Safe': 0, 'At-Risk': 1, 'Imminent': 2}
df['target'] = df['stockout_risk'].map(target_map)

# Drop ID columns, string names, and mostly empty columns
drop_cols = ['store_id', 'sku_id', 'supplier_id', 'date', 'city',
             'city_display', 'sku_name', 'supplier_name',
             'categories_supplied', 'event_name', 'stockout_risk',
             'lead_time_days_actual'] # dropping actual lead time (92% missing)

df_clean = df.drop(columns=drop_cols)

# Convert remaining categoricals to dummy variables (One-Hot Encoding)
df_clean = pd.get_dummies(df_clean, drop_first=True)

# Time-based Split logic (Train: <= 23, Test: >= 24)
train_df = df_clean[df_clean['day'] <= 23]
test_df = df_clean[df_clean['day'] > 23]

# Separate X and y
X_tr = train_df.drop(columns=['target'])
y_tr = train_df['target']

X_te = test_df.drop(columns=['target'])
y_te = test_df['target']

print(f"Training shape: X_tr={X_tr.shape}, y_tr={y_tr.shape}")
print(f"Testing shape: X_te={X_te.shape}, y_te={y_te.shape}")

Training shape: X_tr=(16560, 38), y_tr=(16560,)
Testing shape: X_te=(5040, 38), y_te=(5040,)


### Step 5: Model Building & Evaluation
**Logic:**
- **Baseline:** The majority class is 'Safe' (class 0). Predicting this for every row sets our baseline accuracy.
- **Random Forest:** An ensemble model is chosen because it easily handles non-linear interactions between variables like `reorder_point`, `lead_time`, and demand spikes.
- **Metric Focus:** We are prioritizing the **Recall** metric for the 'Imminent' class (2). In inventory management, the cost of a false negative (missing a stockout) is much higher than a false positive (alerting early).

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Baseline Model (predicting majority class 0)
baseline_preds = [0] * len(y_te)
print("Baseline Accuracy:", round(accuracy_score(y_te, baseline_preds), 4))

# Train Random Forest Classifier
rf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

# Predict and Evaluate
y_pred = rf.predict(X_te)

print("\nRandom Forest Accuracy:", round(accuracy_score(y_te, y_pred), 4))
print("\nClassification Report (Focus on Class 2 - Imminent Recall):")
print(classification_report(y_te, y_pred))

Baseline Accuracy: 0.6228

Random Forest Accuracy: 0.9351

Classification Report (Focus on Class 2 - Imminent Recall):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3139
           1       0.79      0.96      0.87      1126
           2       0.92      0.65      0.76       775

    accuracy                           0.94      5040
   macro avg       0.90      0.87      0.88      5040
weighted avg       0.94      0.94      0.93      5040

